# IEP-2002 — 리랭킹 실험 (Vector + FlashRank)

> **목표**: 벡터 단독 검색 결과에 FlashRank 리랭킹을 추가해 Precision 개선 여부 검증  
> **Baseline**: IEP-1001 벡터 단독 (Recall 0.8537, Precision 0.6117, Faithfulness 0.2748, AR 0.3409)  
> **비교군**: IEP-2001 하이브리드 (Recall 0.5175, Precision 0.6406 — 거절 로직 이슈 있음)  
> **작성일**: 2026-04-27  
> **환경**: Google Colab T4

---

## 실험 설계

```
[IEP-1001 벡터 단독]
질문 → 벡터 검색 TOP_K(3) → Solar 답변

[IEP-2002 벡터 + 리랭킹]
질문 → 벡터 검색 CANDIDATE_K(10) → FlashRank 재정렬 → TOP_K(3) → Solar 답변
```

**거절 로직**: IEP-1001과 완전히 동일 — TOP_K(3) 기준으로 threshold 판단  
→ IEP-2001의 거절 과다 문제(17/100) 재발 방지

## 실행 순서

```
Cell 1   패키지 설치 (런타임 재시작 필요)
Cell 2   경로 설정 ⚠️ 본인 Drive 경로 수정 필요
Cell 3   ChromaDB 로드
Cell 4   FlashRank 모델 로드
Cell 5   리랭킹 함수 정의
Cell 6   단일 질문 비교 테스트 ← 눈으로 먼저 확인
Cell 7   골든 데이터셋 로드
Cell 8   100개 리랭킹 검색 실행
Cell 9   Solar 답변 생성
Cell 10  RAGAS 설정
Cell 11  Context Recall + Precision 측정
Cell 12  Faithfulness 측정
Cell 13  Answer Relevancy 측정
Cell 14  결과 정리 + 3종 비교표 출력
Cell 15  Drive 저장
```

---
## Cell 1 — 패키지 설치

실행 후 **런타임 재시작** 필요 (numpy 바이너리 충돌 방지)

In [ ]:
import subprocess
subprocess.run(["pip", "uninstall", "numpy", "-y"], capture_output=True)
subprocess.run(["pip", "install", "numpy==1.26.4", "--force-reinstall", "-q"], capture_output=True)
print("numpy 재설치 완료")

# FlashRank + RAG 스택
!pip install flashrank --quiet
!pip install chromadb==0.5.11 langchain-community langchain-huggingface \
             sentence-transformers openai python-dotenv ragas --quiet

import numpy as np
print(f"numpy: {np.__version__}")  # 1.26.4 확인
print("설치 완료 → 런타임 재시작 후 Cell 2부터 실행")

# 자동 재시작
import os
os.kill(os.getpid(), 9)

---
## Cell 2 — Google Drive 마운트 + 경로 설정

> ⚠️ **본인 Drive 경로에 맞게 수정하세요**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── ⚠️ 본인 Drive 경로로 수정 ────────────────────────────────────
BASE_DIR    = "/content/drive/MyDrive/YOUR_DATA_DIR"   # 수정 필요
CHROMA_DIR  = f"{BASE_DIR}/chroma_db"
GOLDEN_PATH = f"{BASE_DIR}/golden_dataset.csv"         # 실제 파일명 확인
RESULTS_DIR = f"{BASE_DIR}/results_rerank"
# ─────────────────────────────────────────────────────────────────

COLLECTION_NAME = "ipcc_1001_case3_v1"
EMBED_MODEL     = "jhgan/ko-sroberta-multitask"

# API 키는 Colab Secrets 또는 환경변수로 관리
# 방법 1 (권장): Colab 왼쪽 패널 → Secrets → UPSTAGE_API_KEY 추가
from google.colab import userdata
UPSTAGE_API_KEY = userdata.get('UPSTAGE_API_KEY')

# 방법 2: 직접 입력 (비권장 — 커밋 전 반드시 제거)
# UPSTAGE_API_KEY = ""

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"ChromaDB 존재: {os.path.exists(CHROMA_DIR)}")
print(f"골든 데이터셋 존재: {os.path.exists(GOLDEN_PATH)}")
print(f"결과 폴더: {RESULTS_DIR}")


---
## Cell 3 — ChromaDB 로드

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

print("임베딩 모델 로드 중...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

count = vectorstore._collection.count()
print(f"ChromaDB 로드 완료: {count}개 청크")  # 506 확인

---
## Cell 4 — FlashRank 모델 로드

**모델 선택**
- `ms-marco-MiniLM-L-12-v2`: 영어 기반, 빠름 — 우선 검증용
- `ms-marco-MultiBERT-L-12`: 다국어 지원 — MiniLM 결과가 이상하면 교체

한국어 문서지만 리랭킹 모델이 질문-청크 관련성을 영어 임베딩 공간에서 판단해도  
의미 단위 비교이므로 어느 정도 동작합니다.

In [ ]:
from flashrank import Ranker, RerankRequest

# 우선 MiniLM으로 시작, 결과 이상 시 MultiBERT로 교체
RERANK_MODEL = "ms-marco-MiniLM-L-12-v2"
# RERANK_MODEL = "ms-marco-MultiBERT-L-12"  # 다국어 대안

ranker = Ranker(model_name=RERANK_MODEL)
print(f"FlashRank 로드 완료: {RERANK_MODEL}")

---
## Cell 5 — 리랭킹 함수 정의

**IEP-1001과 거절 로직 완전히 동일**  
→ TOP_K(3)개 기준으로 threshold 판단 (IEP-2001의 CANDIDATE_K 기준 방식 사용 안 함)

```
벡터 검색 10개 뽑기
    ↓
상위 3개(TOP_K)가 모두 threshold 미달이면 거절
    ↓
통과하면 10개 전체를 FlashRank에 투입
    ↓
리랭킹 점수 순으로 상위 3개 반환
```

In [ ]:
from langchain_core.documents import Document

# ── 파라미터 ────────────────────────────────────────────────────────
TOP_K                = 3     # 최종 반환 청크 수
RERANK_CANDIDATE_K   = 10    # 벡터에서 넓게 뽑는 수
SIMILARITY_THRESHOLD = 0.28  # IEP-1001과 동일
# ────────────────────────────────────────────────────────────────────


def rerank_search(question: str):
    """
    벡터 검색 → threshold 필터 (TOP_K 기준, IEP-1001과 동일) → FlashRank 리랭킹
    거절 시 빈 리스트 반환
    """
    # 1. 벡터 검색 — 넓게 뽑기
    v_results = vectorstore.similarity_search_with_relevance_scores(
        question, k=RERANK_CANDIDATE_K
    )

    # 2. threshold 필터 — TOP_K 기준 (IEP-1001과 동일 로직)
    #    상위 3개가 모두 threshold 미달일 때만 거절
    #    IEP-2001과 달리 CANDIDATE_K 전체 기준으로 필터하지 않음
    v_top = v_results[:TOP_K]
    if all(s < SIMILARITY_THRESHOLD for _, s in v_top):
        return []

    # 3. FlashRank 리랭킹 — 10개 전체 투입
    passages = [
        {"id": i, "text": doc.page_content, "meta": doc.metadata}
        for i, (doc, _) in enumerate(v_results)
    ]
    rerank_req = RerankRequest(query=question, passages=passages)
    reranked   = ranker.rerank(rerank_req)

    # 4. 상위 TOP_K 반환
    return [
        (
            Document(
                page_content=passages[r["id"]]["text"],
                metadata=passages[r["id"]]["meta"]
            ),
            r["score"]
        )
        for r in reranked[:TOP_K]
    ]


print("리랭킹 함수 정의 완료")
print(f"  TOP_K={TOP_K}, CANDIDATE_K={RERANK_CANDIDATE_K}, THRESHOLD={SIMILARITY_THRESHOLD}")
print(f"  거절 로직: TOP_K({TOP_K})개 기준 (IEP-1001과 동일)")

---
## Cell 6 — 리랭킹 모델 검증 테스트

**RAGAS 측정 전 필수 확인 단계** — 모델 문제인지 구조 문제인지 먼저 구분합니다.

### 실험 1 — 순위 변화 자동 집계
5개 질문 전체에서 리랭킹 전후 순위가 바뀐 비율을 자동 계산합니다.

### 실험 2 — 사람이 직접 판단
5개 질문에 대해 리랭킹 전후 청크를 나란히 출력합니다. 직접 읽고 판단하세요.

**판단 기준**
```
순위 자주 바뀌고 + 사람이 봐도 더 나음  → 모델 정상 → RAGAS 진행
순위 거의 안 바뀜                        → 변별력 없음 → MultiBERT 교체
순위 바뀌는데 + 사람이 봐도 더 이상함    → 구조 문제 → 방식 재검토
```

In [ ]:
# ── 검증용 질문 5개 ───────────────────────────────────────────────
# 유형별로 골고루 선택 (사실확인 2, 비교 1, 의견예측 1, 경계 1)
verify_questions = [
    ("사실확인", "2011~2020년 전 지구 지표면 온도는 산업화 이전 대비 얼마나 상승했나요?"),
    ("사실확인", "1.5°C 목표를 달성하기 위해 필요한 탄소 감축량은?"),
    ("비교",     "1.5°C와 2°C 온난화 시나리오의 해수면 상승 차이는?"),
    ("의견예측", "기후변화가 생태계에 미치는 영향은?"),
    ("경계케이스","환경 보호를 위해 개인이 할 수 있는 일은?"),
]

# ── 실험 1: 순위 변화 자동 집계 ──────────────────────────────────
print("=" * 65)
print("  실험 1 — 순위 변화 자동 집계")
print("=" * 65)

order_changed = 0
new_page_appeared = 0
rejected_in_verify = 0

results_for_exp2 = []  # 실험 2에서 재사용

for q_type, q in verify_questions:
    # 벡터 단독 (CANDIDATE_K개 뽑기)
    v_results = vectorstore.similarity_search_with_relevance_scores(q, k=RERANK_CANDIDATE_K)

    # 리랭킹
    r_results = rerank_search(q)

    v_pages = [doc.metadata.get("page") for doc, _ in v_results[:TOP_K]]
    r_pages = [doc.metadata.get("page") for doc, _ in r_results] if r_results else []

    changed  = (v_pages != r_pages)
    new_page = bool(set(r_pages) - set(v_pages))
    rejected = not r_results

    if changed:    order_changed     += 1
    if new_page:   new_page_appeared += 1
    if rejected:   rejected_in_verify += 1

    results_for_exp2.append((q_type, q, v_results, r_results))

    status = "거절" if rejected else ("순위변경" if changed else "동일")
    print(f"  [{q_type:6}] {status:6} | 벡터 pages: {v_pages} → 리랭킹 pages: {r_pages}")

print()
print(f"  순위 변경: {order_changed}/{len(verify_questions)}개")
print(f"  새 청크 등장: {new_page_appeared}/{len(verify_questions)}개")
print(f"  거절: {rejected_in_verify}/{len(verify_questions)}개")
print()
if order_changed >= 3:
    print("  ✅ 순위가 자주 바뀜 → 모델이 변별력 있음. 실험 2에서 품질 확인하세요.")
elif order_changed >= 1:
    print("  🟡 일부만 바뀜 → 실험 2에서 실제 품질 확인 필요.")
else:
    print("  ❌ 순위 변화 없음 → 모델 변별력 없음. MultiBERT 교체를 권장합니다.")
    print("     RERANK_MODEL = 'ms-marco-MultiBERT-L-12' 로 Cell 4 수정 후 재실행")

# ── 실험 2: 사람이 직접 판단 ─────────────────────────────────────
print()
print("=" * 65)
print("  실험 2 — 사람이 직접 판단 (아래 청크를 읽고 판단하세요)")
print("=" * 65)

for q_type, q, v_results, r_results in results_for_exp2:
    print(f"""
┌─────────────────────────────────────────────────────────────┐
  [{q_type}] {q}
└─────────────────────────────────────────────────────────────┘""")

    print("  [벡터 단독 TOP_K]")
    for rank, (doc, score) in enumerate(v_results[:TOP_K], 1):
        page = doc.metadata.get("page", "?")
        print(f"    {rank}위 | page={page} | score={score:.4f}")
        print(f"       {doc.page_content[:120].strip().replace(chr(10), ' ')}...")

    if not r_results:
        print("  [리랭킹] → 거절")
    else:
        print("  [리랭킹 TOP_K]")
        for rank, (doc, score) in enumerate(r_results, 1):
            page = doc.metadata.get("page", "?")
            # 벡터 순위 찾기
            v_pages_all = [d.metadata.get("page") for d, _ in v_results]
            v_rank = v_pages_all.index(page) + 1 if page in v_pages_all else "신규"
            print(f"    {rank}위 | page={page} | rerank_score={score:.4f} | (벡터순위: {v_rank}위)")
            print(f"       {doc.page_content[:120].strip().replace(chr(10), ' ')}...")

    print()
    print("  👉 판단: 리랭킹 후 청크가 더 관련 있는가? (직접 읽고 판단)")
    print()


---
## Cell 7 — 골든 데이터셋 로드

> CSV 형식: `user_input`, `reference`, `ID`, `Type` 컬럼

In [ ]:
import csv

with open(GOLDEN_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    golden = list(reader)

print(f"골든 데이터셋: {len(golden)}개")
print(f"키 확인: {list(golden[0].keys())}")
print(f"유형 분포: {{}}".format(
    {t: sum(1 for g in golden if g.get('Type','') == t) for t in ['사실 확인','비교','의견/예측','범위 밖']}
))

---
## Cell 8 — 100개 리랭킹 검색 실행

거절 수 확인 — IEP-1001 수준이어야 정상 (IEP-2001은 17/100으로 과다 거절)

In [ ]:
import time

print("100개 리랭킹 검색 실행 중...")
t0 = time.time()

rerank_dataset = []
rejected_count = 0

for i, item in enumerate(golden):
    q            = item["user_input"]
    ground_truth = item["reference"]

    results  = rerank_search(q)
    contexts = [doc.page_content for doc, _ in results]

    if not contexts:
        rejected_count += 1

    rerank_dataset.append({
        "question":        q,
        "ground_truth":    ground_truth,
        "contexts":        contexts,
        "retrieved_count": len(contexts),
        "rejected":        len(contexts) == 0,
    })

    if (i + 1) % 20 == 0:
        print(f"  {i+1}/100 완료 ({time.time()-t0:.0f}초)")

elapsed = time.time() - t0
avg_chunks = sum(r["retrieved_count"] for r in rerank_dataset) / len(rerank_dataset)

print(f"\n완료: {elapsed:.1f}초")
print(f"거절 수: {rejected_count}/100")
print(f"평균 검색 청크 수: {avg_chunks:.2f}")
print()
print("[거절 수 기준]")
print(f"  IEP-2001 (하이브리드): 17/100  ← 과다 거절")
print(f"  IEP-2002 (리랭킹):    {rejected_count}/100  ← 이 값이 IEP-1001 수준이면 정상")

---
## Cell 9 — Solar 답변 생성 (100개)

In [ ]:
from openai import OpenAI
import time

SOLAR_BASE_URL = "https://api.upstage.ai/v1"
SOLAR_MODEL    = "solar-pro3"

solar_client = OpenAI(
    api_key=UPSTAGE_API_KEY,  # Cell 2에서 설정
    base_url=SOLAR_BASE_URL
)

RAG_PROMPT = """당신은 IPCC AR6 기후변화 보고서 전문 안내 AI입니다.
반드시 아래 제공된 문서 내용만을 근거로 답변하세요.
문서에 없는 내용은 추측하지 말고 "해당 내용은 IPCC 보고서에서 찾을 수 없습니다."라고 답하세요.

[참고 문서]
{context}

[질문]
{question}

[답변]"""

REJECTION_MESSAGE = (
    "죄송합니다. 해당 질문은 IPCC AR6 보고서의 범위를 벗어납니다. "
    "IPCC 기후변화 관련 질문을 입력해 주세요."
)

def generate_answer(question: str, contexts: list) -> str:
    if not contexts:
        return REJECTION_MESSAGE
    context_text = "\n\n".join(f"[청크 {i+1}]\n{c}" for i, c in enumerate(contexts))
    prompt = RAG_PROMPT.format(context=context_text, question=question)
    resp = solar_client.chat.completions.create(
        model=SOLAR_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=800
    )
    return resp.choices[0].message.content


print("Solar 답변 생성 중 (100개)...")
t0 = time.time()

for i, item in enumerate(rerank_dataset):
    item["answer"] = generate_answer(item["question"], item["contexts"])
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/100 완료 ({time.time()-t0:.0f}초)")

print(f"\n답변 생성 완료: {time.time()-t0:.0f}초")

---
## Cell 10 — RAGAS 설정

IEP-2001과 완전히 동일한 조건 (Solar judge, jhgan, strictness=1, batch_size=1)

In [ ]:
import pandas as pd
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from ragas import EvaluationDataset
from langchain_huggingface import HuggingFaceEmbeddings

# ── Baseline 수치 (비교용) ───────────────────────────────────────
IEP1001_RESULTS = {
    "전체":     {"context_recall": 0.8537, "context_precision": 0.6117, "faithfulness": 0.2748, "answer_relevancy": 0.3409},
    "사실 확인": {"context_recall": 0.7381, "context_precision": 0.9133, "faithfulness": 0.5125, "answer_relevancy": 0.5910},
    "비교":     {"context_recall": 0.8600, "context_precision": 0.6733, "faithfulness": 0.2818, "answer_relevancy": 0.2670},
    "의견/예측": {"context_recall": 0.9757, "context_precision": 0.7800, "faithfulness": 0.2619, "answer_relevancy": 0.4342},
    "범위 밖":  {"context_recall": 0.8264, "context_precision": 0.0800, "faithfulness": 0.0774, "answer_relevancy": 0.0714},
}
IEP2001_RESULTS = {
    "전체":     {"context_recall": 0.5175, "context_precision": 0.6406, "faithfulness": 0.2672, "answer_relevancy": 0.3301},
}

# ── Solar judge ──────────────────────────────────────────────────
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model=SOLAR_MODEL,
        openai_api_key=UPSTAGE_API_KEY,   # Cell 2에서 설정
        openai_api_base=SOLAR_BASE_URL,
        temperature=0,
        max_tokens=512,
    )
)
use_llama = False  # llama 미사용 — Solar 단일 judge

# ── jhgan 임베딩 ─────────────────────────────────────────────────
ragas_embeddings_hf = LangchainEmbeddingsWrapper(embeddings)

# ── RunConfig ────────────────────────────────────────────────────
run_config = RunConfig(max_workers=2, timeout=120, max_retries=3)

# ── EvaluationDataset ────────────────────────────────────────────
eval_records = []
for item, r in zip(golden, rerank_dataset):
    eval_records.append({
        "id":                 item.get("ID", ""),
        "type":               item.get("Type", ""),
        "user_input":         r["question"],
        "retrieved_contexts": r["contexts"],
        "response":           r["answer"],
        "reference":          r["ground_truth"],
    })

eval_dataset = EvaluationDataset.from_pandas(pd.DataFrame([{
    "user_input":         r["user_input"],
    "retrieved_contexts": r["retrieved_contexts"],
    "response":           r["response"],
    "reference":          r["reference"],
} for r in eval_records]))

print(f"RAGAS 평가 설정 완료")
print(f"  Judge: {SOLAR_MODEL} (전 지표)")
print(f"  Embeddings: jhgan (AR용)")
print(f"  RunConfig: max_workers=2, timeout=120")
print(f"  batch_size: 1")
print(f"  EvaluationDataset: {len(eval_dataset)}개")

---
## Cell 11 — Context Recall + Precision 측정

judge: Solar (llama 연결 실패 시 대체)  
IEP-1001 Baseline: Recall 0.8537 / Precision 0.6117 (llama 기준 — 직접 비교 주의)

In [ ]:
from ragas import evaluate
from ragas.metrics import ContextRecall, ContextPrecision
import time

print("Context Recall + Precision 측정 중 (Solar judge)...")
t0 = time.time()

result_retrieval = evaluate(
    dataset=eval_dataset,
    metrics=[ContextRecall(), ContextPrecision()],
    llm=ragas_llm,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)

elapsed = time.time() - t0
df_retrieval = result_retrieval.to_pandas()
df_retrieval["id"]   = [r["id"]   for r in eval_records]
df_retrieval["type"] = [r["type"] for r in eval_records]

recall_mean    = df_retrieval["context_recall"].mean(skipna=True)
precision_mean = df_retrieval["context_precision"].mean(skipna=True)
recall_nan     = df_retrieval["context_recall"].isna().sum()
precision_nan  = df_retrieval["context_precision"].isna().sum()

print(f"\n완료: {elapsed:.0f}초")
print(f"  Context Recall    : {recall_mean:.4f}  (NaN: {recall_nan}개)")
print(f"  Context Precision : {precision_mean:.4f}  (NaN: {precision_nan}개)")
print(f"\n  IEP-1001 (llama): Recall 0.8537 / Precision 0.6117 — judge 달라 직접 비교 불가")
print(f"  IEP-2001 (Solar): Recall 0.5175 / Precision 0.6406 — 동일 judge, 비교 가능")

---
## Cell 12 — Faithfulness 측정

judge: Solar  
IEP-1001 Baseline: 0.2748 (NaN 38개) / IEP-2001: 0.2672 (NaN 34개)

In [ ]:
from ragas.metrics import Faithfulness
import time

SAVE_FAITHFULNESS = f"{RESULTS_DIR}/iep2002_faithfulness.csv"

print("Faithfulness 측정 중 (Solar judge, batch_size=1)...")
t0 = time.time()

result_faith = evaluate(
    dataset=eval_dataset,
    metrics=[Faithfulness()],
    llm=ragas_llm,
    embeddings=ragas_embeddings_hf,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)

elapsed = time.time() - t0
df_faith = result_faith.to_pandas()
df_faith["id"]   = [r["id"]   for r in eval_records]
df_faith["type"] = [r["type"] for r in eval_records]
df_faith.to_csv(SAVE_FAITHFULNESS, index=False, encoding="utf-8-sig")

faith_mean         = df_faith["faithfulness"].mean(skipna=True)
faith_nan          = df_faith["faithfulness"].isna().sum()
faith_conservative = round(faith_mean * (100 - faith_nan) / 100, 4)

print(f"\n완료: {elapsed:.0f}초")
print(f"  Faithfulness : {faith_mean:.4f}  (NaN: {faith_nan}개)")
print(f"  보수적 수치  : {faith_conservative:.4f}")
print(f"  IEP-1001     : 0.2748 (NaN 38개)  diff: {faith_mean-0.2748:+.4f}")
print(f"  IEP-2001     : 0.2672 (NaN 34개)  diff: {faith_mean-0.2672:+.4f}")
print(f"  저장: {SAVE_FAITHFULNESS}")

---
## Cell 13 — Answer Relevancy 측정

judge: Solar + jhgan + strictness=1  
IEP-1001 Baseline: 0.3409 / IEP-2001: 0.3301

In [ ]:
from ragas.metrics import AnswerRelevancy
import time

SAVE_RELEVANCY = f"{RESULTS_DIR}/iep2002_relevancy.csv"
metric_ar = AnswerRelevancy(strictness=1)

print("Answer Relevancy 측정 중 (Solar + jhgan + strictness=1)...")
t0 = time.time()

result_ar = evaluate(
    dataset=eval_dataset,
    metrics=[metric_ar],
    llm=ragas_llm,
    embeddings=ragas_embeddings_hf,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)

elapsed = time.time() - t0
df_ar = result_ar.to_pandas()
df_ar["id"]   = [r["id"]   for r in eval_records]
df_ar["type"] = [r["type"] for r in eval_records]
df_ar.to_csv(SAVE_RELEVANCY, index=False, encoding="utf-8-sig")

ar_mean = df_ar["answer_relevancy"].mean(skipna=True)
ar_nan  = df_ar["answer_relevancy"].isna().sum()

print(f"\n완료: {elapsed:.0f}초")
print(f"  Answer Relevancy : {ar_mean:.4f}  (NaN: {ar_nan}개)")
print(f"  IEP-1001         : 0.3409  diff: {ar_mean-0.3409:+.4f}")
print(f"  IEP-2001         : 0.3301  diff: {ar_mean-0.3301:+.4f}")
print(f"  저장: {SAVE_RELEVANCY}")

---
## Cell 14 — 결과 정리 + 3종 비교표

**IEP-1001 (벡터 단독) vs IEP-2001 (하이브리드) vs IEP-2002 (리랭킹)**

In [ ]:
import numpy as np
from datetime import datetime

TYPES       = ["사실 확인", "비교", "의견/예측", "범위 밖"]
ALL_METRICS = ["context_recall", "context_precision", "faithfulness", "answer_relevancy"]

df_raw = pd.DataFrame({
    "id":                [r["id"]   for r in eval_records],
    "type":              [r["type"] for r in eval_records],
    "user_input":        [r["user_input"] for r in eval_records],
    "context_recall":    df_retrieval["context_recall"].values,
    "context_precision": df_retrieval["context_precision"].values,
    "faithfulness":      df_faith["faithfulness"].values,
    "answer_relevancy":  df_ar["answer_relevancy"].values,
})

overall_mean = df_raw[ALL_METRICS].mean(skipna=True).round(4)
overall_nan  = df_raw[ALL_METRICS].isna().sum()
summary      = df_raw.groupby("type")[ALL_METRICS].mean().round(4)

faith_conservative = round(overall_mean["faithfulness"] * (100 - overall_nan["faithfulness"]) / 100, 4)
ar_conservative    = round(overall_mean["answer_relevancy"] * (100 - overall_nan["answer_relevancy"]) / 100, 4)

# ── 3종 비교표 ───────────────────────────────────────────────────
print("=" * 72)
print("  3종 비교: 벡터 단독 vs 하이브리드 vs 리랭킹 (Solar judge 기준)")
print("=" * 72)
comparison = pd.DataFrame([
    {
        "방식":            "① 벡터 단독 (IEP-1001)*",
        "Recall":          0.8537, "Precision": 0.6117,
        "Faithfulness":    0.2748, "Faith보수적": 0.1704,
        "AR":              0.3409, "거절수": "?",
    },
    {
        "방식":            "② 하이브리드 (IEP-2001)",
        "Recall":          0.5175, "Precision": 0.6406,
        "Faithfulness":    0.2672, "Faith보수적": 0.1764,
        "AR":              0.3301, "거절수": "17/100",
    },
    {
        "방식":            "③ 리랭킹 (IEP-2002)",
        "Recall":          overall_mean["context_recall"],
        "Precision":       overall_mean["context_precision"],
        "Faithfulness":    overall_mean["faithfulness"],
        "Faith보수적":     faith_conservative,
        "AR":              overall_mean["answer_relevancy"],
        "거절수":          f"{rejected_count}/100",
    },
])
print(comparison.to_string(index=False))
print("\n*IEP-1001: llama judge 기준 — 수치 직접 비교 불가, 방향성 참고용")

# ── 유형별 상세 ──────────────────────────────────────────────────
print("\n[유형별 상세 — IEP-2002 vs IEP-1001]")
for m in ALL_METRICS:
    print(f"\n  {m}")
    print(f"    {'유형':<10} {'IEP-2002':>10} {'IEP-1001':>10} {'차이':>8}")
    print(f"    {'-'*42}")
    for t in TYPES:
        v2 = summary.loc[t, m] if t in summary.index else float("nan")
        v1 = IEP1001_RESULTS.get(t, {}).get(m, float("nan"))
        diff = f"{v2-v1:+.4f}" if not (np.isnan(v2) or np.isnan(v1)) else "   N/A"
        print(f"    {t:<10} {v2:>10.4f} {v1:>10.4f} {diff:>8}")
    print(f"    {'-'*42}")
    print(f"    {'전체':<10} {overall_mean[m]:>10.4f} {IEP1001_RESULTS['전체'][m]:>10.4f} {overall_mean[m]-IEP1001_RESULTS['전체'][m]:>+8.4f}")

# ── 생존 편향 보정 ────────────────────────────────────────────────
print("\n[생존 편향 보정]")
print(f"  {'지표':<22} {'낙관적':>10} {'유효샘플':>8} {'보수적':>10}")
print(f"  {'-'*54}")
for m, pes in [("faithfulness", faith_conservative), ("answer_relevancy", ar_conservative)]:
    vn = 100 - overall_nan[m]
    print(f"  {m:<22} {overall_mean[m]:>10.4f} {vn:>8}개 {pes:>10.4f}")

# ── README 마크다운 ──────────────────────────────────────────────
print("\n[README 마크다운]")
md  = "| 유형 | Context Recall | Context Precision | Faithfulness | Answer Relevancy |\n"
md += "| :--- | :---: | :---: | :---: | :---: |\n"
for t in TYPES:
    cr = summary.loc[t, "context_recall"]    if t in summary.index else float("nan")
    cp = summary.loc[t, "context_precision"] if t in summary.index else float("nan")
    fa = summary.loc[t, "faithfulness"]      if t in summary.index else float("nan")
    ar = summary.loc[t, "answer_relevancy"]  if t in summary.index else float("nan")
    md += f"| {t} | {cr:.4f} | {cp:.4f} | {fa:.4f} | {ar:.4f} |\n"
md += (f"| **전체** | **{overall_mean['context_recall']:.4f}** "
       f"| **{overall_mean['context_precision']:.4f}** "
       f"| **{overall_mean['faithfulness']:.4f}** "
       f"| **{overall_mean['answer_relevancy']:.4f}** |\n")
print(md)

---
## Cell 15 — Drive 저장

In [ ]:
import json, os
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# raw CSV
raw_path = f"{RESULTS_DIR}/iep2002_raw_{timestamp}.csv"
df_raw.to_csv(raw_path, index=False, encoding="utf-8-sig")

# summary CSV
df_s = summary.copy()
df_s.loc["전체"] = overall_mean
summary_path = f"{RESULTS_DIR}/iep2002_summary_{timestamp}.csv"
df_s.to_csv(summary_path, encoding="utf-8-sig")

# dataset JSON (contexts 포함)
dataset_path = f"{RESULTS_DIR}/iep2002_dataset_{timestamp}.json"
with open(dataset_path, "w", encoding="utf-8") as f:
    json.dump(rerank_dataset, f, ensure_ascii=False, indent=2)

print("✅ 저장 완료")
print(f"  raw     → {raw_path}")
print(f"  summary → {summary_path}")
print(f"  dataset → {dataset_path}")
print(f"\n⚠️ 주석")
print(f"  Faithfulness: Solar judge, NaN {overall_nan['faithfulness']}개 — 보수적 {faith_conservative}")
print(f"  Answer Relevancy: strictness=1, jhgan 임베딩")
print(f"  Recall/Precision: Solar judge (IEP-1001 llama 기준과 직접 비교 불가)")

---
## 실험 완료 후 memory.md 업데이트 내용 (복사용)

```
## IEP-2002 리랭킹 결과 (날짜 기입)

| 지표               | IEP-1001 벡터 | IEP-2001 하이브리드 | IEP-2002 리랭킹 |
| :---               | :---:         | :---:               | :---:           |
| Context Recall     | 0.8537*       | 0.5175              | [결과 기입]     |
| Context Precision  | 0.6117*       | 0.6406              | [결과 기입]     |
| Faithfulness       | 0.2748        | 0.2672              | [결과 기입]     |
| Answer Relevancy   | 0.3409        | 0.3301              | [결과 기입]     |
| 거절 수            | ?             | 17/100              | [결과 기입]     |

*IEP-1001: llama judge 기준

리랭킹 모델: ms-marco-MiniLM-L-12-v2
거절 로직: TOP_K(3) 기준 (IEP-1001과 동일)
```